# S3-04: 건축공학 구조 도구 종합 실습
**모멘트 계산, 전단 검토, 지진하중 계산, 통합 설계 검토**

## 학습 목표
- RC 보 공칭 휨 모멘트 계산 도구를 Tool Use로 완전히 구현한다
- RC 보 전단 강도 검토 도구를 구현하고 설계 적정성을 판정한다
- 등가정적해석법 지진하중 계산 도구를 구현한다
- 여러 구조 도구를 통합하여 종합 설계 검토 시스템을 구축한다

## 사전 준비
`.env` 파일에 API 키가 설정되어 있어야 합니다.

> **이 노트북의 모든 실습은 건축공학 도메인입니다.**

In [ ]:
%pip install anthropic python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
import json
import math

client = Anthropic()
model = "claude-sonnet-4-0"
print("설정 완료")

In [ ]:
# Tool Use 루프 함수 (공통)
def run_tool_loop(user_message, tools, tool_map, system=None):
    """Tool Use 루프를 실행하고 최종 텍스트 응답을 반환한다."""
    messages = [{"role": "user", "content": user_message}]
    
    for i in range(10):
        params = {"model": model, "max_tokens": 4096, "tools": tools, "messages": messages}
        if system:
            params["system"] = system
        
        response = client.messages.create(**params)
        
        if response.stop_reason == "end_turn":
            return "".join(b.text for b in response.content if b.type == "text")
        elif response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    func = tool_map.get(block.name)
                    try:
                        result = func(**block.input) if func else {"error": f"Unknown: {block.name}"}
                        tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(result, ensure_ascii=False)})
                        print(f"  [{block.name}] -> {json.dumps(result, ensure_ascii=False)[:120]}")
                    except Exception as e:
                        tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(e), "is_error": True})
                        print(f"  [{block.name}] ERROR: {e}")
            messages.append({"role": "user", "content": tool_results})
        else:
            break
    return "최대 반복 초과"

---
## 실습 1: RC 보 휨 강도 계산 도구

### 과제: KDS 14 20 20 등가직사각형 응력 블록 방법으로 공칭 휨 모멘트 Mn을 계산하는 도구를 구현하세요.

**요구사항:**
1. 함수: `calc_flexural_strength(b, d, fck, fy, As)` → dict
2. beta1 결정: fck<=28이면 0.85, 28<fck<=56이면 0.85-0.007*(fck-28), fck>56이면 0.65
3. a = As*fy / (0.85*fck*b), c = a/beta1
4. Mn = As*fy*(d - a/2) / 1e6 (kN-m)
5. epsilon_t = 0.003*(d-c)/c로 강도감소계수 phi 결정
6. 도구 스키마 + Tool Use 루프로 완전한 검토 수행

**검증 조건:**
- 350x600 보, fck=27MPa, fy=400MPa, 5-D25(2540mm2)
- 예상: Mn 약 550-570 kN-m, phi*Mn 약 460-490 kN-m

In [ ]:
# TODO: 휨 강도 계산 도구 구현

def calc_flexural_strength(b, d, fck, fy, As):
    """RC 보 공칭 휨 모멘트 강도 (KDS 14 20 20)"""
    pass  # 구현하세요

# TODO: 도구 스키마 + Tool Use 루프

In [ ]:
# ===== 정답 =====

def calc_flexural_strength(b, d, fck, fy, As):
    """RC 보 공칭 휨 모멘트 강도 (KDS 14 20 20)"""
    # beta1 결정
    if fck <= 28:
        beta1 = 0.85
    elif fck <= 56:
        beta1 = 0.85 - 0.007 * (fck - 28)
    else:
        beta1 = 0.65
    
    # 등가직사각형 응력 블록
    a = (As * fy) / (0.85 * fck * b)  # mm
    c = a / beta1  # mm
    
    # 공칭 모멘트 강도
    Mn = As * fy * (d - a / 2) / 1e6  # kN-m
    
    # 인장 철근 변형률
    epsilon_t = 0.003 * (d - c) / c
    
    # 강도감소계수
    if epsilon_t >= 0.005:
        phi = 0.85  # 인장지배
    elif epsilon_t <= 0.002:
        phi = 0.65  # 압축지배
    else:
        phi = 0.65 + (epsilon_t - 0.002) * (200 / 3)  # 전이구간
    
    rho = As / (b * d)
    
    return {
        "a_mm": round(a, 1),
        "c_mm": round(c, 1),
        "beta1": round(beta1, 3),
        "Mn_kNm": round(Mn, 1),
        "phi": round(phi, 3),
        "phi_Mn_kNm": round(phi * Mn, 1),
        "epsilon_t": round(epsilon_t, 5),
        "rho": round(rho, 4),
        "section_type": "인장지배" if epsilon_t >= 0.005 else "전이구간" if epsilon_t > 0.002 else "압축지배"
    }

flexural_tool = {
    "name": "calc_flexural_strength",
    "description": "RC 보의 공칭 휨 모멘트 강도 Mn과 설계 모멘트 강도 phi*Mn을 KDS 14 20 20 등가직사각형 응력블록으로 계산한다.",
    "input_schema": {
        "type": "object",
        "properties": {
            "b": {"type": "number", "description": "보 너비 (mm)"},
            "d": {"type": "number", "description": "유효 깊이 (mm)"},
            "fck": {"type": "number", "description": "콘크리트 설계기준강도 (MPa)"},
            "fy": {"type": "number", "description": "철근 항복강도 (MPa)"},
            "As": {"type": "number", "description": "인장 철근 단면적 (mm2)"}
        },
        "required": ["b", "d", "fck", "fy", "As"]
    }
}

# Tool Use 루프로 실행
result_text = run_tool_loop(
    "350x600 보의 휨 강도를 계산해줘. fck=27MPa, fy=400MPa, 인장철근 5-D25 (As=2540mm2). 설계 모멘트 Mu=380kN-m에 대해 적합한지 판정해줘.",
    [flexural_tool],
    {"calc_flexural_strength": calc_flexural_strength}
)
print(f"\n최종 응답:\n{result_text}")

def verify():
    r = calc_flexural_strength(350, 600, 27, 400, 2540)
    assert 500 < r["Mn_kNm"] < 600, f"Mn 범위 확인: {r['Mn_kNm']}"
    assert r["beta1"] == 0.85, f"fck=27이면 beta1=0.85: {r['beta1']}"
    assert r["epsilon_t"] > 0.005, f"인장지배 확인: {r['epsilon_t']}"
    assert r["phi"] == 0.85, f"인장지배이면 phi=0.85: {r['phi']}"
    assert r["phi_Mn_kNm"] > 380, f"phi*Mn > Mu(380) 확인: {r['phi_Mn_kNm']}"
    print(f"모든 검증 통과! Mn={r['Mn_kNm']}kN-m, phi*Mn={r['phi_Mn_kNm']}kN-m > Mu=380kN-m → OK")

verify()

---
## 실습 2: RC 보 전단 강도 검토 도구

### 과제: KDS 14 20 22 기준으로 전단 강도를 검토하는 도구를 구현하세요.

**요구사항:**
1. 함수: `calc_shear_strength(b, d, fck, fy, Av, s, Vu)` → dict
2. Vc = (1/6)*sqrt(fck)*b*d / 1000 (kN)
3. Vs = Av*fy*d/s / 1000 (kN)
4. phi_Vn = 0.75*(Vc+Vs)
5. 최소 전단보강근, 최대 간격도 검토
6. Tool Use 루프로 검토

**검증 조건:**
- 350x600 보, fck=27MPa, fy=400MPa
- D10@200 (Av=142.6mm2), Vu=280kN

In [ ]:
# TODO: 전단 강도 검토 도구 구현

In [ ]:
# ===== 정답 =====

def calc_shear_strength(b, d, fck, fy, Av, s, Vu):
    """RC 보 전단 강도 검토 (KDS 14 20 22)"""
    # 콘크리트 전단 강도 (간략법)
    Vc = (1/6) * math.sqrt(fck) * b * d / 1000  # kN
    
    # 전단 보강근 강도
    Vs = Av * fy * d / s / 1000  # kN
    
    # 설계 전단 강도
    phi = 0.75
    phi_Vn = phi * (Vc + Vs)
    
    # 최소 전단보강근 (KDS 14 20 22)
    Av_min = max(
        0.062 * math.sqrt(fck) * b * s / fy,
        0.35 * b * s / fy
    )
    
    # 최대 간격 검토
    s_max = min(d / 2, 600)  # mm
    
    # 판정
    strength_ok = phi_Vn >= Vu
    av_ok = Av >= Av_min
    spacing_ok = s <= s_max
    
    return {
        "Vc_kN": round(Vc, 1),
        "Vs_kN": round(Vs, 1),
        "phi_Vn_kN": round(phi_Vn, 1),
        "Vu_kN": Vu,
        "DCR": round(Vu / phi_Vn, 3),
        "strength_check": "OK" if strength_ok else "NG",
        "Av_min_mm2": round(Av_min, 1),
        "Av_provided_mm2": Av,
        "min_rebar_check": "OK" if av_ok else "NG",
        "s_max_mm": s_max,
        "s_provided_mm": s,
        "spacing_check": "OK" if spacing_ok else "NG",
        "overall": "PASS" if (strength_ok and av_ok and spacing_ok) else "FAIL"
    }

shear_tool = {
    "name": "calc_shear_strength",
    "description": "RC 보의 전단 강도를 KDS 14 20 22 기준으로 검토한다. Vc, Vs, phi*Vn, 최소보강근, 최대간격을 검토한다.",
    "input_schema": {
        "type": "object",
        "properties": {
            "b": {"type": "number", "description": "보 너비 (mm)"},
            "d": {"type": "number", "description": "유효 깊이 (mm)"},
            "fck": {"type": "number", "description": "콘크리트 설계기준강도 (MPa)"},
            "fy": {"type": "number", "description": "철근 항복강도 (MPa)"},
            "Av": {"type": "number", "description": "전단보강근 단면적 (mm2, 양다리)"},
            "s": {"type": "number", "description": "전단보강근 간격 (mm)"},
            "Vu": {"type": "number", "description": "설계 전단력 (kN)"}
        },
        "required": ["b", "d", "fck", "fy", "Av", "s", "Vu"]
    }
}

result_text = run_tool_loop(
    "350x600 보의 전단 설계를 검토해줘. fck=27MPa, fy=400MPa, D10@200(Av=142.6mm2), Vu=280kN. 최소보강근과 간격도 검토해줘.",
    [shear_tool],
    {"calc_shear_strength": calc_shear_strength}
)
print(f"\n최종 응답:\n{result_text}")

def verify():
    r = calc_shear_strength(350, 600, 27, 400, 142.6, 200, 280)
    assert r["Vc_kN"] > 0, "Vc > 0"
    assert r["Vs_kN"] > 0, "Vs > 0"
    assert r["phi_Vn_kN"] > 0, "phi_Vn > 0"
    assert r["DCR"] > 0, "DCR > 0"
    assert r["overall"] in ("PASS", "FAIL")
    assert r["spacing_check"] == "OK", f"간격 200 < d/2=300: {r['spacing_check']}"
    print(f"모든 검증 통과! Vc={r['Vc_kN']}kN, Vs={r['Vs_kN']}kN, phi*Vn={r['phi_Vn_kN']}kN, 판정: {r['overall']}")

verify()

---
## 실습 3: 등가정적해석법 지진하중 계산 도구

### 과제: KDS 41 17 00에 따른 밑면 전단력 V를 계산하는 도구를 구현하세요.

**요구사항:**
1. 함수: `calc_seismic_base_shear(W, S, site_class, I, R, T)` → dict
2. 지반증폭계수 Fa, Fv 테이블 (간략화)
3. SDS = (2/3)*Fa*S, SD1 = (2/3)*Fv*S
4. Cs = SDS/(R/I) 또는 SD1/(T*(R/I)) (주기에 따라)
5. V = Cs * W
6. Tool Use 루프로 실행

**검증 조건:**
- W=50000kN, S=0.22, 지반 S3, I=1.2, R=5.0, T=0.8sec

In [ ]:
# TODO: 지진하중 계산 도구 구현

In [ ]:
# ===== 정답 =====

def calc_seismic_base_shear(W, S, site_class, I, R, T):
    """등가정적해석법 밑면 전단력 V (KDS 41 17 00)"""
    # 지반증폭계수 (간략화 테이블)
    Fa_table = {
        "S1": {0.11: 1.12, 0.22: 0.90},
        "S2": {0.11: 1.40, 0.22: 1.00},
        "S3": {0.11: 1.70, 0.22: 1.10},
        "S4": {0.11: 2.00, 0.22: 1.30},
        "S5": {0.11: 2.40, 0.22: 1.50},
    }
    Fv_table = {
        "S1": {0.11: 1.12, 0.22: 0.84},
        "S2": {0.11: 1.56, 0.22: 1.17},
        "S3": {0.11: 2.28, 0.22: 1.71},
        "S4": {0.11: 3.42, 0.22: 2.56},
        "S5": {0.11: 4.56, 0.22: 3.42},
    }
    
    # 가장 가까운 S 값 선택
    S_key = 0.22 if S >= 0.165 else 0.11
    Fa = Fa_table.get(site_class, {}).get(S_key, 1.0)
    Fv = Fv_table.get(site_class, {}).get(S_key, 1.0)
    
    # 설계 스펙트럼 가속도
    SDS = (2/3) * Fa * S
    SD1 = (2/3) * Fv * S
    
    # 설계 지진력 계수 Cs
    if T <= SD1 / SDS:
        Cs = SDS / (R / I)  # 단주기
    else:
        Cs = SD1 / (T * (R / I))  # 장주기
    
    Cs = max(Cs, 0.01)  # 최소값
    
    # 밑면 전단력
    V = Cs * W
    
    return {
        "Fa": Fa,
        "Fv": Fv,
        "SDS": round(SDS, 4),
        "SD1": round(SD1, 4),
        "Cs": round(Cs, 5),
        "V_kN": round(V, 1),
        "W_kN": W,
        "T_sec": T,
        "method": "KDS 41 17 00 등가정적해석법"
    }

seismic_tool = {
    "name": "calc_seismic_base_shear",
    "description": "KDS 41 17 00 등가정적해석법에 의한 밑면 전단력 V를 계산한다. W, S, 지반종류, I, R, T를 입력받는다.",
    "input_schema": {
        "type": "object",
        "properties": {
            "W": {"type": "number", "description": "건물 유효 중량 (kN)"},
            "S": {"type": "number", "description": "지역계수 (0.22 또는 0.11)"},
            "site_class": {"type": "string", "description": "지반종류", "enum": ["S1","S2","S3","S4","S5"]},
            "I": {"type": "number", "description": "중요도계수"},
            "R": {"type": "number", "description": "반응수정계수"},
            "T": {"type": "number", "description": "고유주기 (sec)"}
        },
        "required": ["W", "S", "site_class", "I", "R", "T"]
    }
}

result_text = run_tool_loop(
    "다음 건물의 지진하중을 계산해줘. W=50000kN, S=0.22(I구역), 지반 S3, I=1.2(중요), R=5.0(RC 모멘트골조), T=0.8sec",
    [seismic_tool],
    {"calc_seismic_base_shear": calc_seismic_base_shear}
)
print(f"\n최종 응답:\n{result_text}")

def verify():
    r = calc_seismic_base_shear(50000, 0.22, "S3", 1.2, 5.0, 0.8)
    assert r["Fa"] == 1.10, f"S3/0.22 Fa=1.10: {r['Fa']}"
    assert r["Fv"] == 1.71, f"S3/0.22 Fv=1.71: {r['Fv']}"
    assert r["SDS"] > 0, "SDS > 0"
    assert r["SD1"] > 0, "SD1 > 0"
    assert r["V_kN"] > 0, "V > 0"
    assert 1000 < r["V_kN"] < 10000, f"V가 합리적 범위: {r['V_kN']}"
    print(f"모든 검증 통과! SDS={r['SDS']}, SD1={r['SD1']}, Cs={r['Cs']}, V={r['V_kN']}kN")

verify()

---
## 실습 4: 통합 구조 설계 검토 시스템

### 과제: 휨 + 전단 + 철근비를 통합 검토하는 다중 도구 시스템을 구축하세요.

**요구사항:**
1. 위에서 만든 `calc_flexural_strength`, `calc_shear_strength` + 새로운 `check_reinforcement`
2. 3개 도구를 동시에 등록
3. Claude에게 종합 검토를 요청하면 필요한 도구를 자동 선택
4. 모든 검토 결과를 종합하여 최종 판정

**검토 조건:**
- 보 B1: 350 x 600 mm
- fck = 27 MPa, fy = 400 MPa
- 인장 철근: 5-D25 (As = 2,540 mm2)
- 전단 보강근: D10@200 (Av = 142.6 mm2)
- Mu = 380 kN-m, Vu = 250 kN

In [ ]:
# TODO: 3개 도구 통합 + 종합 검토 시스템

In [ ]:
# ===== 정답 =====

# 도구 함수 3: 철근비 검토
def check_reinforcement(b, d, fck, fy, As):
    """RC 보 철근비 적정성 검토 (KDS 14 20 20)"""
    rho = As / (b * d)
    rho_min = max(0.25 * math.sqrt(fck) / fy, 1.4 / fy)
    # 최대 철근비 (인장지배 한계)
    beta1 = 0.85 if fck <= 28 else max(0.65, 0.85 - 0.007 * (fck - 28))
    rho_b = 0.85 * beta1 * fck / fy * (0.003 / (0.003 + 0.002))
    rho_max = 0.75 * rho_b
    
    return {
        "rho": round(rho, 4),
        "rho_min": round(rho_min, 4),
        "rho_max": round(rho_max, 4),
        "min_check": "OK" if rho >= rho_min else "NG (철근 부족)",
        "max_check": "OK" if rho <= rho_max else "NG (과다 배근)",
        "overall": "OK" if rho_min <= rho <= rho_max else "NG"
    }

rebar_tool = {
    "name": "check_reinforcement",
    "description": "RC 보의 철근비가 최소/최대 기준 이내인지 검토한다 (KDS 14 20 20).",
    "input_schema": {
        "type": "object",
        "properties": {
            "b": {"type": "number", "description": "보 너비 (mm)"},
            "d": {"type": "number", "description": "유효 깊이 (mm)"},
            "fck": {"type": "number", "description": "콘크리트 설계기준강도 (MPa)"},
            "fy": {"type": "number", "description": "철근 항복강도 (MPa)"},
            "As": {"type": "number", "description": "인장 철근 단면적 (mm2)"}
        },
        "required": ["b", "d", "fck", "fy", "As"]
    }
}

# 3개 도구 통합 등록
all_tools = [flexural_tool, shear_tool, rebar_tool]
all_tool_map = {
    "calc_flexural_strength": calc_flexural_strength,
    "calc_shear_strength": calc_shear_strength,
    "check_reinforcement": check_reinforcement,
}

system_prompt = (
    "당신은 KDS 14 20 기준에 정통한 구조공학 전문가입니다. "
    "도구를 사용하여 정확한 계산을 수행하고, 결과를 전문적으로 해석합니다. "
    "종합 검토 시 휨, 전단, 철근비를 모두 검토하고 표 형식으로 정리합니다."
)

final_result = run_tool_loop(
    (
        "다음 RC 보의 종합 설계 검토를 수행해줘.\n\n"
        "보 B1: 350 x 600 mm\n"
        "콘크리트: fck = 27 MPa\n"
        "철근: fy = 400 MPa\n"
        "인장 철근: 5-D25 (As = 2,540 mm2)\n"
        "전단 보강근: D10@200 (양다리, Av = 142.6 mm2)\n"
        "설계 하중: Mu = 380 kN-m, Vu = 250 kN\n\n"
        "1) 휨 강도 검토\n"
        "2) 전단 강도 검토\n"
        "3) 철근비 적정성 검토\n"
        "4) 종합 판정"
    ),
    all_tools,
    all_tool_map,
    system=system_prompt
)

print(f"\n{'='*60}")
print("종합 설계 검토 결과:")
print(f"{'='*60}")
print(final_result)

def verify():
    # 각 도구 함수 직접 검증
    flex = calc_flexural_strength(350, 600, 27, 400, 2540)
    shear = calc_shear_strength(350, 600, 27, 400, 142.6, 200, 250)
    rebar = check_reinforcement(350, 600, 27, 400, 2540)
    
    assert flex["phi_Mn_kNm"] > 380, f"phi*Mn > Mu(380): {flex['phi_Mn_kNm']}"
    assert shear["phi_Vn_kN"] > 250 or shear["overall"] in ("PASS","FAIL"), "전단 검토 완료"
    assert rebar["overall"] in ("OK", "NG"), "철근비 검토 완료"
    
    assert isinstance(final_result, str) and len(final_result) > 100, "종합 검토 결과가 충분히 상세해야 함"
    
    print("\n=== 검증 결과 ===")
    print(f"  휨: phi*Mn={flex['phi_Mn_kNm']}kN-m > Mu=380kN-m → {'OK' if flex['phi_Mn_kNm'] > 380 else 'NG'}")
    print(f"  전단: phi*Vn={shear['phi_Vn_kN']}kN > Vu=250kN → {shear['strength_check']}")
    print(f"  철근비: rho={rebar['rho']} ({rebar['rho_min']} ~ {rebar['rho_max']}) → {rebar['overall']}")
    print("모든 검증 통과! 통합 구조 검토 시스템 완성!")

verify()